# Gold Aggregation — Tablas Gold
**Proyecto 2 FHBD | Capa Gold — Ejecución Manual**

Equivalente manual del Task 3 del DAG.

Genera las siguientes tablas Gold en Iceberg usando **MERGE/upsert**:

| # | Tabla | Descripción |
|---|---|---|
| ★ | `cant_post_x_user_hist` | Posts por usuario, año y tipo (pipeline) |
| 2 | `vote_stats_per_post` | Votos +/- por post |
| 3 | `top_tags` | Ranking de etiquetas más usadas |
| 4 | `user_engagement` | Interacciones totales por usuario |
| 5 | `badges_summary` | Resumen de insignias por usuario |

---
⚠️ Pre-requisito: todas las tablas Silver deben existir.

**Fuentes Silver:**
- `nessie.silver.post_hist`
- `nessie.silver.users_hist`
- `nessie.silver.votes_hist`
- `nessie.silver.badges_hist`

## 1. Configuración

In [ ]:
import os
from datetime import datetime

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT',  'http://proyecto2-minio:9000')
MINIO_ACCESS   = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET   = os.getenv('MINIO_SECRET_KEY', 'minioadmin123')
NESSIE_URL     = os.getenv('NESSIE_URL', 'http://proyecto2-nessie:19120/api/v1')
NESSIE_BRANCH  = os.getenv('NESSIE_BRANCH', 'main')
SILVER_NS      = os.getenv('SILVER_NAMESPACE', 'silver')
GOLD_NS        = os.getenv('GOLD_NAMESPACE', 'gold')
SPARK_MASTER   = os.getenv('SPARK_MASTER', 'spark://spark-master:7077')
FECHA_CARGUE   = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')

print(f'Nessie   : {NESSIE_URL}')
print(f'Silver NS: nessie.{SILVER_NS}')
print(f'Gold NS  : nessie.{GOLD_NS}')
print(f'Spark    : {SPARK_MASTER}')
print(f'Fecha    : {FECHA_CARGUE}')

## 2. Crear SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('gold_agg_notebook')
    .master(SPARK_MASTER)
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,'
            'org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
    .config('spark.sql.catalog.nessie',              'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
    .config('spark.sql.catalog.nessie.uri',          NESSIE_URL)
    .config('spark.sql.catalog.nessie.ref',          NESSIE_BRANCH)
    .config('spark.sql.catalog.nessie.authentication.type', 'NONE')
    .config('spark.sql.catalog.nessie.warehouse',    's3a://iceberg/')
    .config('spark.hadoop.fs.s3a.endpoint',          MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key',        MINIO_ACCESS)
    .config('spark.hadoop.fs.s3a.secret.key',        MINIO_SECRET)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl',              'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)

print(f'✅ SparkSession creada — versión {spark.version}')

## 3. Verificar tablas Silver disponibles

In [ ]:
silver_tables = ['post_hist', 'users_hist', 'votes_hist', 'badges_hist']

for t in silver_tables:
    try:
        count = spark.sql(f'SELECT COUNT(*) as c FROM nessie.{SILVER_NS}.{t}').collect()[0]['c']
        print(f'  ✅ nessie.{SILVER_NS}.{t}: {count:,} filas')
    except Exception as e:
        print(f'  ❌ nessie.{SILVER_NS}.{t}: {e}')

## 4. Helper: write_gold_merge

In [ ]:
def write_gold_merge(df, gold_table: str, merge_key: str):
    """Crea la tabla Gold si no existe y ejecuta MERGE/upsert."""
    full_table = f'nessie.{GOLD_NS}.{gold_table}'

    try:
        spark.sql(f'CREATE NAMESPACE IF NOT EXISTS nessie.{GOLD_NS}')
    except Exception:
        pass

    try:
        spark.sql(f'DESCRIBE TABLE {full_table}')
    except Exception:
        df.limit(0).writeTo(full_table) \
            .tableProperty('write.format.default', 'parquet') \
            .tableProperty('write.metadata.compression-codec', 'gzip') \
            .createOrReplace()

    df.createOrReplaceTempView(f'src_{gold_table}')
    cols = [c for c in df.columns if c != merge_key]
    set_clause = ',\n            '.join([f'target.{c} = source.{c}' for c in cols])

    spark.sql(f"""
        MERGE INTO {full_table} AS target
        USING src_{gold_table} AS source
        ON target.{merge_key} = source.{merge_key}
        WHEN MATCHED THEN UPDATE SET {set_clause}
        WHEN NOT MATCHED THEN INSERT *
    """)

    total = spark.sql(f'SELECT COUNT(*) as c FROM {full_table}').collect()[0]['c']
    print(f'  ✅ {full_table} → {total:,} filas')
    return total

print('✅ Helper write_gold_merge listo')

## 5. ★ cant_post_x_user_hist (tabla principal del pipeline)

In [ ]:
df = spark.sql(f"""
    SELECT
        p.OwnerUserId                           AS user_id,
        COALESCE(u.DisplayName, 'Unknown')      AS display_name,
        p.anio                                  AS anio,
        p.PostTypeId                            AS post_type_id,
        CASE
            WHEN p.PostTypeId = 1 THEN 'Question'
            WHEN p.PostTypeId = 2 THEN 'Answer'
            ELSE 'Other'
        END                                     AS post_type,
        COUNT(p.Id)                             AS cant_posts,
        COALESCE(SUM(p.Score), 0)               AS total_score,
        ROUND(AVG(p.Score), 2)                  AS avg_score,
        COALESCE(SUM(p.ViewCount), 0)           AS total_views,
        CAST('{FECHA_CARGUE}' AS TIMESTAMP)     AS fecha_cargue
    FROM nessie.{SILVER_NS}.post_hist p
    LEFT JOIN nessie.{SILVER_NS}.users_hist u ON p.OwnerUserId = u.Id
    WHERE p.OwnerUserId IS NOT NULL
    GROUP BY p.OwnerUserId, u.DisplayName, p.anio, p.PostTypeId
""")

df = df.withColumn('merge_key',
    F.concat_ws('_',
        F.col('user_id').cast('string'),
        F.col('anio').cast('string'),
        F.col('post_type_id').cast('string')
    )
)

print('[★] cant_post_x_user_hist')
write_gold_merge(df, 'cant_post_x_user_hist', 'merge_key')

print('\nMuestra:')
spark.sql(f'SELECT * FROM nessie.{GOLD_NS}.cant_post_x_user_hist LIMIT 5').show(truncate=False)

## 6. vote_stats_per_post

In [ ]:
df = spark.sql(f"""
    SELECT
        p.Id                                                AS post_id,
        COALESCE(p.Title, '')                               AS title,
        p.PostTypeId                                        AS post_type_id,
        p.OwnerUserId                                       AS owner_user_id,
        p.anio                                              AS anio,
        COUNT(v.Id)                                         AS total_votes,
        SUM(CASE WHEN v.VoteTypeId = 2 THEN 1 ELSE 0 END)  AS upvotes,
        SUM(CASE WHEN v.VoteTypeId = 3 THEN 1 ELSE 0 END)  AS downvotes,
        SUM(CASE WHEN v.VoteTypeId = 2 THEN 1
                 WHEN v.VoteTypeId = 3 THEN -1 ELSE 0 END) AS net_votes,
        COALESCE(p.Score, 0)                                AS score,
        COALESCE(p.ViewCount, 0)                            AS view_count,
        CAST('{FECHA_CARGUE}' AS TIMESTAMP)                 AS fecha_cargue
    FROM nessie.{SILVER_NS}.post_hist p
    LEFT JOIN nessie.{SILVER_NS}.votes_hist v ON p.Id = v.PostId
    GROUP BY p.Id, p.Title, p.PostTypeId, p.OwnerUserId, p.anio, p.Score, p.ViewCount
""")

print('[2] vote_stats_per_post')
write_gold_merge(df, 'vote_stats_per_post', 'post_id')
spark.sql(f'SELECT * FROM nessie.{GOLD_NS}.vote_stats_per_post ORDER BY total_votes DESC LIMIT 5').show()

## 7. top_tags

In [ ]:
df = spark.sql(f"""
    SELECT
        trim(tag)                               AS tag,
        anio,
        COUNT(*)                                AS cant_preguntas,
        COALESCE(SUM(Score), 0)                 AS total_score,
        ROUND(AVG(Score), 2)                    AS avg_score,
        COALESCE(SUM(ViewCount), 0)             AS total_views,
        COALESCE(SUM(AnswerCount), 0)           AS total_answers,
        CAST('{FECHA_CARGUE}' AS TIMESTAMP)     AS fecha_cargue
    FROM nessie.{SILVER_NS}.post_hist
    LATERAL VIEW explode(
        split(regexp_replace(Tags, '[<>]', ' '), '\\\\s+')
    ) t AS tag
    WHERE PostTypeId = 1
      AND Tags IS NOT NULL AND Tags != ''
      AND trim(tag) != ''
    GROUP BY trim(tag), anio
    ORDER BY cant_preguntas DESC
""")

df = df.withColumn('merge_key',
    F.concat_ws('_', F.col('tag'), F.col('anio').cast('string'))
)

print('[3] top_tags')
write_gold_merge(df, 'top_tags', 'merge_key')
spark.sql(f'SELECT tag, anio, cant_preguntas FROM nessie.{GOLD_NS}.top_tags ORDER BY cant_preguntas DESC LIMIT 10').show()

## 8. user_engagement

In [ ]:
df = spark.sql(f"""
    SELECT
        u.Id                                            AS user_id,
        COALESCE(u.DisplayName, 'Unknown')              AS display_name,
        COALESCE(u.Reputation, 0)                       AS reputation,
        COALESCE(u.Location, '')                        AS location,
        COUNT(DISTINCT p.Id)                            AS total_posts,
        SUM(CASE WHEN p.PostTypeId = 1 THEN 1 ELSE 0 END) AS total_questions,
        SUM(CASE WHEN p.PostTypeId = 2 THEN 1 ELSE 0 END) AS total_answers,
        COALESCE(SUM(p.Score), 0)                       AS total_score,
        COALESCE(SUM(p.ViewCount), 0)                   AS total_views,
        COALESCE(SUM(p.CommentCount), 0)                AS total_comments,
        COUNT(DISTINCT v.Id)                            AS total_votes_received,
        COUNT(DISTINCT b.Id)                            AS total_badges,
        CAST('{FECHA_CARGUE}' AS TIMESTAMP)             AS fecha_cargue
    FROM nessie.{SILVER_NS}.users_hist u
    LEFT JOIN nessie.{SILVER_NS}.post_hist p ON u.Id = p.OwnerUserId
    LEFT JOIN nessie.{SILVER_NS}.votes_hist v ON p.Id = v.PostId
    LEFT JOIN nessie.{SILVER_NS}.badges_hist b ON u.Id = b.UserId
    GROUP BY u.Id, u.DisplayName, u.Reputation, u.Location
""")

print('[4] user_engagement')
write_gold_merge(df, 'user_engagement', 'user_id')
spark.sql(f"""
    SELECT user_id, display_name, total_posts, total_badges, reputation
    FROM nessie.{GOLD_NS}.user_engagement
    ORDER BY total_posts DESC LIMIT 5
""").show()

## 9. badges_summary

In [ ]:
df = spark.sql(f"""
    SELECT
        b.UserId                                            AS user_id,
        COALESCE(u.DisplayName, 'Unknown')                  AS display_name,
        COALESCE(u.Reputation, 0)                           AS reputation,
        COUNT(b.Id)                                         AS total_badges,
        SUM(CASE WHEN b.Class = 1 THEN 1 ELSE 0 END)       AS gold_badges,
        SUM(CASE WHEN b.Class = 2 THEN 1 ELSE 0 END)       AS silver_badges,
        SUM(CASE WHEN b.Class = 3 THEN 1 ELSE 0 END)       AS bronze_badges,
        SUM(CASE WHEN b.TagBased = true THEN 1 ELSE 0 END) AS tag_based_badges,
        COUNT(DISTINCT b.Name)                              AS unique_badge_types,
        CAST('{FECHA_CARGUE}' AS TIMESTAMP)                 AS fecha_cargue
    FROM nessie.{SILVER_NS}.badges_hist b
    LEFT JOIN nessie.{SILVER_NS}.users_hist u ON b.UserId = u.Id
    GROUP BY b.UserId, u.DisplayName, u.Reputation
""")

print('[5] badges_summary')
write_gold_merge(df, 'badges_summary', 'user_id')
spark.sql(f"""
    SELECT display_name, total_badges, gold_badges, silver_badges, bronze_badges
    FROM nessie.{GOLD_NS}.badges_summary
    ORDER BY total_badges DESC LIMIT 5
""").show()

## 10. Resumen final

In [ ]:
gold_tables = [
    'cant_post_x_user_hist',
    'vote_stats_per_post',
    'top_tags',
    'user_engagement',
    'badges_summary',
]

print('=' * 55)
print('RESUMEN TABLAS GOLD')
print('=' * 55)
for t in gold_tables:
    try:
        count = spark.sql(f'SELECT COUNT(*) as c FROM nessie.{GOLD_NS}.{t}').collect()[0]['c']
        print(f'  ✅ nessie.{GOLD_NS}.{t}: {count:,} filas')
    except Exception as e:
        print(f'  ❌ nessie.{GOLD_NS}.{t}: {e}')

spark.stop()
print('\n✅ Gold completado — listo para consultar en Dremio')